In [1]:
import polars as pl
cad = pl.read_csv("complete_aggregation2.csv").select([
    pl.col("player").alias("id"),
    "rounds_played"
])

metap1 = pl.read_csv("esea_meta_demos.part1.csv")
metap2 = pl.read_csv("esea_meta_demos.part2.csv")

meta = pl.concat([metap1, metap2]).select(
    ["file", "map", "round", "winner_side"]


dmgp1 = pl.read_csv("esea_master_dmg_demos.part1.csv")
dmgp2 = pl.read_csv("esea_master_dmg_demos.part2.csv")

dmg = pl.concat([dmgp1, dmgp2]).select([
    "file", "round", "att_side", "vic_side", "att_id", "vic_id"
])



# remove duplicate interactions
dmg = dmg.unique([
    "file", "round", "att_side", "vic_side", "att_id", "vic_id"
])

# ATTACH ROUND WINNER
dmg = dmg.join(
    meta.select(["file", "round", "winner_side"]),
    on=["file", "round"],
    how="left"
)

# PLAYER-ROUND TABLE
att = dmg.select([
    "file",
    "round",
    pl.col("att_id").alias("id"),
    pl.col("att_side").alias("side"),
    "winner_side"
])

vic = dmg.select([
    "file",
    "round",
    pl.col("vic_id").alias("id"),
    pl.col("vic_side").alias("side"),
    "winner_side"
])

players = (
    pl.concat([att, vic])
    .filter(pl.col("id").is_not_null())
    .unique(["file", "round", "id", "side"])
    .with_columns(
        (pl.col("side") == pl.col("winner_side")).alias("won_round")
    )
)

# 
# ROUND WINS PER PLAYER
player_round_wins = (
    players
    .group_by("id")
    .agg(
        pl.sum("won_round").alias("rounds_won")
    )
)


player_wins = (
    cad
    .join(player_round_wins, on="id", how="left")
    .with_columns(
        pl.col("rounds_won").fill_null(0)
    )
    .with_columns(
        (pl.col("rounds_won") / pl.col("rounds_played")).alias("win_rate")
    )
)



bad_round_counts = player_wins.filter(
    pl.col("rounds_won") > pl.col("rounds_played")
)

print("players with rounds_won > rounds_played:")
print(bad_round_counts)

print("count:", bad_round_counts.height)

match_winners = (
    meta
    .group_by(["file", "winner_side"])
    .agg(pl.len().alias("side_round_wins"))
    .sort(["file", "side_round_wins"], descending=[False, True])
    .group_by("file")
    .agg([
        pl.first("winner_side").alias("match_winner_side"),
        pl.first("side_round_wins").alias("match_winner_rounds")
    ])
)

# one row per player per file
player_matches = (
    players
    .group_by(["file", "id"])
    .agg(
        pl.first("side").alias("side")
    )
    .join(match_winners, on="file", how="left")
    .with_columns(
        (pl.col("side") == pl.col("match_winner_side")).alias("won_match")
    )
)

player_match_wins = (
    player_matches
    .group_by("id")
    .agg([
        pl.sum("won_match").alias("match_wins"),
        pl.len().alias("matches_played")
    ])
    .with_columns(
        (pl.col("match_wins") / pl.col("matches_played")).alias("match_win_rate")
    )
)

print(player_wins)
print(player_match_wins)

players with rounds_won > rounds_played:
shape: (0, 4)
┌─────┬───────────────┬────────────┬──────────┐
│ id  ┆ rounds_played ┆ rounds_won ┆ win_rate │
│ --- ┆ ---           ┆ ---        ┆ ---      │
│ i64 ┆ i64           ┆ u32        ┆ f64      │
╞═════╪═══════════════╪════════════╪══════════╡
└─────┴───────────────┴────────────┴──────────┘
count: 0
shape: (23_979, 4)
┌───────────────────┬───────────────┬────────────┬──────────┐
│ id                ┆ rounds_played ┆ rounds_won ┆ win_rate │
│ ---               ┆ ---           ┆ ---        ┆ ---      │
│ i64               ┆ i64           ┆ u32        ┆ f64      │
╞═══════════════════╪═══════════════╪════════════╪══════════╡
│ 76561197960269146 ┆ 533           ┆ 251        ┆ 0.470919 │
│ 76561197960269231 ┆ 28            ┆ 12         ┆ 0.428571 │
│ 76561197960269266 ┆ 210           ┆ 99         ┆ 0.471429 │
│ 76561197960269474 ┆ 326           ┆ 161        ┆ 0.493865 │
│ 76561197960269573 ┆ 328           ┆ 139        ┆ 0.42378  │
│ …      

In [8]:
import polars as pl


cad = pl.read_csv("complete_aggregation2.csv")

# if needed, align key name
cad_full = cad.rename({"player": "id"})


extra_ids = player_match_wins.join(
    player_wins.select("id"),
    on="id",
    how="anti"
).select("id")

player_match_wins_clean = player_match_wins.join(
    extra_ids,
    on="id",
    how="anti"
)


final = (
    cad_full
    .join(
        player_wins.select(["id", "rounds_won", "win_rate"]),
        on="id",
        how="left"
    )
    .join(
        player_match_wins_clean.select(["id", "match_wins", "matches_played", "match_win_rate"]),
        on="id",
        how="left"
    )
)

# rename id back to player if you want original schema
final = final.rename({"id": "player"})


bad_round_counts = final.filter(
    pl.col("rounds_won") > pl.col("rounds_played")
)

print("rows where rounds_won > rounds_played:")
print(bad_round_counts)
print("count:", bad_round_counts.height)


final.write_csv("complete_aggregation3.csv")
print("saved to complete_aggregation3.csv")

rows where rounds_won > rounds_played:
shape: (0, 19)
┌────────┬──────────────────┬────────────┬──────────────┬───┬──────────┬────────────┬────────────────┬────────────────┐
│ player ┆ damage_per_round ┆ net_damage ┆ total_damage ┆ … ┆ win_rate ┆ match_wins ┆ matches_played ┆ match_win_rate │
│ ---    ┆ ---              ┆ ---        ┆ ---          ┆   ┆ ---      ┆ ---        ┆ ---            ┆ ---            │
│ i64    ┆ f64              ┆ f64        ┆ f64          ┆   ┆ f64      ┆ u32        ┆ u32            ┆ f64            │
╞════════╪══════════════════╪════════════╪══════════════╪═══╪══════════╪════════════╪════════════════╪════════════════╡
└────────┴──────────────────┴────────────┴──────────────┴───┴──────────┴────────────┴────────────────┴────────────────┘
count: 0
saved to complete_aggregation3.csv


In [7]:
null_summary = final.select([
    pl.all().null_count()
])

print(null_summary)

shape: (1, 19)
┌────────┬──────────────────┬────────────┬──────────────┬───┬──────────┬────────────┬────────────────┬────────────────┐
│ player ┆ damage_per_round ┆ net_damage ┆ total_damage ┆ … ┆ win_rate ┆ match_wins ┆ matches_played ┆ match_win_rate │
│ ---    ┆ ---              ┆ ---        ┆ ---          ┆   ┆ ---      ┆ ---        ┆ ---            ┆ ---            │
│ u32    ┆ u32              ┆ u32        ┆ u32          ┆   ┆ u32      ┆ u32        ┆ u32            ┆ u32            │
╞════════╪══════════════════╪════════════╪══════════════╪═══╪══════════╪════════════╪════════════════╪════════════════╡
│ 0      ┆ 0                ┆ 0          ┆ 0            ┆ … ┆ 0        ┆ 0          ┆ 0              ┆ 0              │
└────────┴──────────────────┴────────────┴──────────────┴───┴──────────┴────────────┴────────────────┴────────────────┘
